In [ ]:
import ROOT
import math
import os

# --- PDG Masses (GeV) for initialization ---
M_Y1S_PDG = 9.46030
M_Y2S_PDG = 10.02326
M_Y3S_PDG = 10.3552

# --- STARlight Cross Section Ratios (at mid-rapidity y~0) ---
R2S_STARLIGHT = 0.445
R3S_STARLIGHT = 0.315

# --- Double Sided Crystal Ball Function ---
def DoubleSidedCrystalBall(x, par):
    N = par[0]
    mu = par[1]
    sigma = par[2]
    alphaL = abs(par[3])
    nL = par[4]
    alphaH = abs(par[5])
    nH = par[6]

    t = (x[0] - mu) / sigma

    if t < -alphaL:
        A = math.pow(nL / alphaL, nL) * math.exp(-0.5 * alphaL * alphaL)
        B = nL / alphaL - alphaL
        return N * A * math.pow(B - t, -nL)
    elif t > alphaH:
        A = math.pow(nH / alphaH, nH) * math.exp(-0.5 * alphaH * alphaH)
        B = nH / alphaH - alphaH
        return N * A * math.pow(B + t, -nH)
    else:
        return N * math.exp(-0.5 * t * t)

# --- Background Function (Exponential) ---
def Background(x, par):
    return par[0] * math.exp(par[1] * x[0])

# --- Total Fit Function ---
def TotalFitFunction(x, par):
    # Y(1S)
    par1S = [par[0], par[1], par[2], par[3], par[4], par[5], par[6]]
    y1s = DoubleSidedCrystalBall(x, par1S)

    # Y(2S) 
    par2S = [par[7], par[8], par[9], par[3], par[4], par[5], par[6]]
    y2s = DoubleSidedCrystalBall(x, par2S)

    # Y(3S) 
    par3S = [par[10], par[11], par[12], par[3], par[4], par[5], par[6]]
    y3s = DoubleSidedCrystalBall(x, par3S)
    
    # Background
    bkg = Background(x, [par[13], par[14]])

    return y1s + y2s + y3s + bkg

def FitUpsilon_Improved():
    ROOT.gStyle.SetOptFit(0)
    ROOT.gStyle.SetOptStat(0)

    # 1. Load Data
    filename = "~/merged_elecs_test.root"
    f = ROOT.TFile.Open(filename)
    if not f or f.IsZombie():
        print(f"Error: Could not open file {filename}!")
        return

    hInvMass = f.Get("h_invMass_Reco_All")
    if not hInvMass:
        print("Error: Could not find h_invMass_Reco_All")
        return

    # 2. Setup Fit Function
    fitFunc = ROOT.TF1("fitFunc", TotalFitFunction, 8.0, 12.0, 15)
    fitFunc.SetNpx(2000)
    fitFunc.SetLineColor(ROOT.kRed)
    fitFunc.SetLineWidth(2)

    peakHeight = hInvMass.GetMaximum()
    
    # Initialization
    fitFunc.SetParName(3, "Alpha L"); fitFunc.SetParameter(3, 1.2); fitFunc.SetParLimits(3, 0.1, 5.0)
    fitFunc.SetParName(4, "n L");     fitFunc.SetParameter(4, 2.0); fitFunc.SetParLimits(4, 1.0, 20.0)
    fitFunc.SetParName(5, "Alpha H"); fitFunc.SetParameter(5, 1.5); fitFunc.SetParLimits(5, 0.1, 5.0)
    fitFunc.SetParName(6, "n H");     fitFunc.SetParameter(6, 2.0); fitFunc.SetParLimits(6, 1.0, 20.0)

    fitFunc.SetParName(0, "Norm 1S");  fitFunc.SetParameter(0, peakHeight)
    fitFunc.SetParName(1, "Mean 1S");  fitFunc.SetParameter(1, M_Y1S_PDG); fitFunc.SetParLimits(1, 9.3, 9.6)
    fitFunc.SetParName(2, "Sigma 1S"); fitFunc.SetParameter(2, 0.05);      fitFunc.SetParLimits(2, 0.01, 0.15)

    fitFunc.SetParName(7, "Norm 2S");  fitFunc.SetParameter(7, peakHeight * 0.5) 
    fitFunc.SetParName(8, "Mean 2S");  fitFunc.SetParameter(8, M_Y2S_PDG); fitFunc.SetParLimits(8, 9.9, 10.15)
    fitFunc.SetParName(9, "Sigma 2S"); fitFunc.SetParameter(9, 0.06);      fitFunc.SetParLimits(9, 0.01, 0.15)

    fitFunc.SetParName(10, "Norm 3S");  fitFunc.SetParameter(10, peakHeight * 0.3) 
    fitFunc.SetParName(11, "Mean 3S");  fitFunc.SetParameter(11, M_Y3S_PDG); fitFunc.SetParLimits(11, 10.2, 10.5)
    fitFunc.SetParName(12, "Sigma 3S"); fitFunc.SetParameter(12, 0.06);      fitFunc.SetParLimits(12, 0.01, 0.15)

    fitFunc.SetParName(13, "Bkg Norm");  fitFunc.SetParameter(13, 10)
    fitFunc.SetParName(14, "Bkg Slope"); fitFunc.SetParameter(14, -0.1)

    # 3. Perform Fit
    print("Fitting with improved model...")
    hInvMass.Fit("fitFunc", "RM") 

    # 4. Extract Individual Components
    f1S = ROOT.TF1("f1S", DoubleSidedCrystalBall, 8.0, 12.0, 7)
    f1S.SetParameters(fitFunc.GetParameter(0), fitFunc.GetParameter(1), fitFunc.GetParameter(2),
                      fitFunc.GetParameter(3), fitFunc.GetParameter(4), fitFunc.GetParameter(5), fitFunc.GetParameter(6))

    f2S = ROOT.TF1("f2S", DoubleSidedCrystalBall, 8.0, 12.0, 7)
    f2S.SetParameters(fitFunc.GetParameter(7), fitFunc.GetParameter(8), fitFunc.GetParameter(9),
                      fitFunc.GetParameter(3), fitFunc.GetParameter(4), fitFunc.GetParameter(5), fitFunc.GetParameter(6))

    f3S = ROOT.TF1("f3S", DoubleSidedCrystalBall, 8.0, 12.0, 7)
    f3S.SetParameters(fitFunc.GetParameter(10), fitFunc.GetParameter(11), fitFunc.GetParameter(12),
                      fitFunc.GetParameter(3), fitFunc.GetParameter(4), fitFunc.GetParameter(5), fitFunc.GetParameter(6))

    # --- INTEGRAL SCALING CALCULATIONS ---
    bin_width = hInvMass.GetBinWidth(1)
    
    # Calculate raw yields
    yield_1s = f1S.Integral(8.0, 12.0) / bin_width
    yield_2s = f2S.Integral(8.0, 12.0) / bin_width
    yield_3s = f3S.Integral(8.0, 12.0) / bin_width
    
    # Scale the 2S and 3S yields down according to STARlight ratios
    scaled_yield_2s = yield_2s * R2S_STARLIGHT
    scaled_yield_3s = yield_3s * R3S_STARLIGHT
    
    print("\n--- Yield Integration & Scaling Results ---")
    print(f"Y(1S) Raw Yield:    {yield_1s:.2f}")
    print(f"Y(2S) Raw Yield:    {yield_2s:.2f}  | Scaled (x{R2S_STARLIGHT}): {scaled_yield_2s:.2f}")
    print(f"Y(3S) Raw Yield:    {yield_3s:.2f}  | Scaled (x{R3S_STARLIGHT}): {scaled_yield_3s:.2f}")
    print("-------------------------------------------\n")

    # 5. Visualization Setup
    c1 = ROOT.TCanvas("c1", "Improved Upsilon Fit", 900, 700)
    hInvMass.SetTitle("Invariant Mass Fit of #Upsilon(nS) #rightarrow e^{+}e^{-}; m_{e^{+}e^{-}} (GeV/c^{2}); Events / 20 MeV")
    
    hInvMass.GetYaxis().SetRangeUser(0, peakHeight * 1.35)
    hInvMass.GetXaxis().SetRangeUser(8.0, 12.0)
    
    hInvMass.SetMarkerStyle(20)
    hInvMass.SetLineColor(ROOT.kBlack)
    hInvMass.Draw("PE")
    fitFunc.Draw("SAME")

    # Drawing components
    f1S.SetLineColor(ROOT.kAzure+2); f1S.SetLineStyle(2); f1S.SetLineWidth(2)
    f1S.SetFillColorAlpha(ROOT.kAzure+2, 0.15)
    f1S.Draw("SAME FC"); f1S.Draw("SAME L")

    f2S.SetLineColor(ROOT.kMagenta+1); f2S.SetLineStyle(2); f2S.SetLineWidth(2)
    f2S.SetFillColorAlpha(ROOT.kMagenta+1, 0.15)
    f2S.Draw("SAME FC"); f2S.Draw("SAME L")

    f3S.SetLineColor(ROOT.kGreen+2); f3S.SetLineStyle(2); f3S.SetLineWidth(2)
    f3S.SetFillColorAlpha(ROOT.kGreen+2, 0.15)
    f3S.Draw("SAME FC"); f3S.Draw("SAME L")
    
    fBkg = ROOT.TF1("fBkg", Background, 8.0, 12.0, 2)
    fBkg.SetParameters(fitFunc.GetParameter(13), fitFunc.GetParameter(14))
    fBkg.SetLineColor(ROOT.kGray+2); fBkg.SetLineStyle(7); fBkg.SetLineWidth(2)
    fBkg.Draw("SAME")
    
    hInvMass.Draw("PE SAME")

    leg = ROOT.TLegend(0.65, 0.55, 0.88, 0.88)
    leg.SetBorderSize(0)
    leg.AddEntry(hInvMass, "Simulation Data", "pe")
    leg.AddEntry(fitFunc, "Total Fit", "l")
    leg.AddEntry(f1S, "#Upsilon(1S) Signal", "f")
    leg.AddEntry(f2S, "#Upsilon(2S) Signal", "f")
    leg.AddEntry(f3S, "#Upsilon(3S) Signal", "f")
    leg.AddEntry(fBkg, "Background", "l")
    leg.Draw()

    # Results Box (Includes Integrals + Scaled Output)
    pt = ROOT.TPaveText(0.12, 0.40, 0.50, 0.88, "NDC")
    pt.SetFillColor(0)
    pt.SetBorderSize(0)
    pt.SetTextAlign(12)
    pt.SetTextSize(0.025)
    
    pt.AddText("DSCB Fit Results & Scaled Yields:")
    
    # Y(1S) Stats
    pt.AddText(f"#Upsilon(1S) M = {fitFunc.GetParameter(1):.3f} #pm {fitFunc.GetParError(1):.3f} GeV")
    pt.AddText(f"#Upsilon(1S) #sigma = {fitFunc.GetParameter(2)*1000.0:.1f} MeV  |  Yield = {int(yield_1s)}")
    
    pt.AddText("") 
    
    # Y(2S) Stats
    pt.AddText(f"#Upsilon(2S) M = {fitFunc.GetParameter(8):.3f} #pm {fitFunc.GetParError(8):.3f} GeV")
    pt.AddText(f"#Upsilon(2S) #sigma = {fitFunc.GetParameter(9)*1000.0:.1f} MeV")
    pt.AddText(f"#Upsilon(2S) Raw = {int(yield_2s)}  |  Scaled = {int(scaled_yield_2s)}")
    
    pt.AddText("") 
    
    # Y(3S) Stats 
    pt.AddText(f"#Upsilon(3S) M = {fitFunc.GetParameter(11):.3f} #pm {fitFunc.GetParError(11):.3f} GeV")
    pt.AddText(f"#Upsilon(3S) #sigma = {fitFunc.GetParameter(12)*1000.0:.1f} MeV")
    pt.AddText(f"#Upsilon(3S) Raw = {int(yield_3s)}  |  Scaled = {int(scaled_yield_3s)}")
    
    pt.Draw()

    #os.makedirs("outputs/plots", exist_ok=True)
    c1.SaveAs("Fit_Inclusive_Upsilon_DSCB_Scaled.pdf")
    ROOT.gROOT.SetSelectedPad(c1)

if __name__ == "__main__":
    FitUpsilon_Improved()

In [ ]:
import ROOT
import math
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

# Use standard ROOT style for matplotlib
hep.style.use(hep.style.ROOT)

# --- PDG Masses (GeV) for initialization ---
M_Y1S_PDG = 9.46030
M_Y2S_PDG = 10.02326
M_Y3S_PDG = 10.3552

# --- Branching-ratio handling for histogram weights ---
BR_Y1S_EE = 0.0234
BR_Y2S_EE = 0.0191
BR_Y3S_EE = 0.0218

# Options:
#   "visible_decay" keeps the tuple weights as visible e+e- yields
#   "inclusive_upsilon" multiplies each state by 1 / BR(Υ -> e+e-)
BR_WEIGHT_MODE = "visible_decay"

# --- Double Sided Crystal Ball Function ---
def DoubleSidedCrystalBall(x, par):
    N = par[0]
    mu = par[1]
    sigma = par[2]
    alphaL = abs(par[3])
    nL = par[4]
    alphaH = abs(par[5])
    nH = par[6]

    t = (x[0] - mu) / sigma

    if t < -alphaL:
        A = math.pow(nL / alphaL, nL) * math.exp(-0.5 * alphaL * alphaL)
        B = nL / alphaL - alphaL
        return N * A * math.pow(B - t, -nL)
    elif t > alphaH:
        A = math.pow(nH / alphaH, nH) * math.exp(-0.5 * alphaH * alphaH)
        B = nH / alphaH - alphaH
        return N * A * math.pow(B + t, -nH)
    else:
        return N * math.exp(-0.5 * t * t)

# --- Background Function (Exponential) ---
def Background(x, par):
    return par[0] * math.exp(par[1] * x[0])

# --- Total Fit Function ---
def TotalFitFunction(x, par):
    # Y(1S)
    par1S = [par[0], par[1], par[2], par[3], par[4], par[5], par[6]]
    y1s = DoubleSidedCrystalBall(x, par1S)

    # Y(2S) 
    par2S = [par[7], par[8], par[9], par[3], par[4], par[5], par[6]]
    y2s = DoubleSidedCrystalBall(x, par2S)

    # Y(3S) 
    par3S = [par[10], par[11], par[12], par[3], par[4], par[5], par[6]]
    y3s = DoubleSidedCrystalBall(x, par3S)
    
    # Background
    bkg = Background(x, [par[13], par[14]])

    return y1s + y2s + y3s + bkg

def draw_epic_labels(ax):
    """Draws the standard ePIC performance block in the top left corner."""
    # We use ax.text with transform=ax.transAxes to position it relative to the axes 
    # (0,1 is top left). We add a little padding so it doesn't touch the border.
    
    # 1. ePIC Performance (Bold)
    ax.text(0.04, 0.94, "ePIC Performance", transform=ax.transAxes, 
            fontsize=12, fontweight='bold', ha='left', va='top')
    
    # 2. Campaign
    ax.text(0.04, 0.88, "26.04.1 Campaign", transform=ax.transAxes, 
            fontsize=11, ha='left', va='top')
    
    # 3. Reaction and Energy
    ax.text(0.04, 0.82, r"e+p $\Upsilon_{\gamma p}$ 10x100 GeV$^2$", transform=ax.transAxes, 
            fontsize=11, ha='left', va='top')
    
    # 4. Luminosity and sqrt(s)
    ax.text(0.04, 0.76, r"$\mathcal{L}_{proj} = 10$ fb$^{-1}$, $\sqrt{s} = 63$ GeV", transform=ax.transAxes, 
            fontsize=11, ha='left', va='top')

def FitUpsilon_Improved():
    # 1. Load Data using RDataFrame (Exclusive tuples)
    filename = "~/merged_elecs_test_10t100_10x100.root" #"~/merged_elecs_test_.root"
    
    try:
        df = ROOT.RDataFrame("UpsilonTree", filename)
    except Exception as e:
        print(f"Error opening file {filename}: {e}")
        return

    # =========================================================================
    # INCLUSIVE PHYSICS SELECTION
    # =========================================================================
    print("Applying inclusive selection logic...")
    df_signal = df.Filter("reco_mass >= 8.0 && reco_mass <= 12.0") \
                  .Filter("reco_W >= 0.0")
    
    # 2. Setup Histogram Dynamically
    print("Generating weighted invariant mass histogram...")
    # 200 bins between 8.0 and 12.0 results in exactly 20 MeV per bin
    hModel = ROOT.RDF.TH1DModel("h_invMass_Reco_All", "Title", 200, 8.0, 12.0)
    
    # Apply optional branching-ratio correction in the histogram weight.
    # Use true_mass to assign the state, since this is MC.
    if BR_WEIGHT_MODE == "inclusive_upsilon":
        br_corr_expr = (
            f"(fabs(true_mass - {M_Y1S_PDG}) < 0.20) ? (1.0/{BR_Y1S_EE}) : "
            f"(fabs(true_mass - {M_Y2S_PDG}) < 0.20) ? (1.0/{BR_Y2S_EE}) : "
            f"(fabs(true_mass - {M_Y3S_PDG}) < 0.20) ? (1.0/{BR_Y3S_EE}) : "
            "1.0"
        )
        df_signal = df_signal.Define("br_corr", br_corr_expr)
        df_signal = df_signal.Define("fit_weight", "weight * br_corr")
    else:
        df_signal = df_signal.Define("fit_weight", "weight")

    hInvMass_ptr = df_signal.Histo1D(hModel, "reco_mass", "fit_weight")
    hInvMass = hInvMass_ptr.GetValue()

    if not hInvMass or hInvMass.GetEntries() == 0:
        print("Error: Histogram is empty after applying the physics selection.")
        return

    # 3. Setup Fit Function
    fitFunc = ROOT.TF1("fitFunc", TotalFitFunction, 8.0, 12.0, 15)
    
    peakHeight = hInvMass.GetMaximum()
    
    # Initialization
    fitFunc.SetParName(3, "Alpha L"); fitFunc.SetParameter(3, 1.2); fitFunc.SetParLimits(3, 0.1, 5.0)
    fitFunc.SetParName(4, "n L");     fitFunc.SetParameter(4, 2.0); fitFunc.SetParLimits(4, 1.0, 20.0)
    fitFunc.SetParName(5, "Alpha H"); fitFunc.SetParameter(5, 1.5); fitFunc.SetParLimits(5, 0.1, 5.0)
    fitFunc.SetParName(6, "n H");     fitFunc.SetParameter(6, 2.0); fitFunc.SetParLimits(6, 1.0, 20.0)

    fitFunc.SetParName(0, "Norm 1S");  fitFunc.SetParameter(0, peakHeight)
    fitFunc.SetParName(1, "Mean 1S");  fitFunc.SetParameter(1, M_Y1S_PDG); fitFunc.SetParLimits(1, 9.3, 9.6)
    fitFunc.SetParName(2, "Sigma 1S"); fitFunc.SetParameter(2, 0.05);      fitFunc.SetParLimits(2, 0.01, 0.15)

    fitFunc.SetParName(7, "Norm 2S");  fitFunc.SetParameter(7, peakHeight * 0.5) 
    fitFunc.SetParName(8, "Mean 2S");  fitFunc.SetParameter(8, M_Y2S_PDG); fitFunc.SetParLimits(8, 9.9, 10.15)
    fitFunc.SetParName(9, "Sigma 2S"); fitFunc.SetParameter(9, 0.06);      fitFunc.SetParLimits(9, 0.01, 0.15)

    fitFunc.SetParName(10, "Norm 3S");  fitFunc.SetParameter(10, peakHeight * 0.3) 
    fitFunc.SetParName(11, "Mean 3S");  fitFunc.SetParameter(11, M_Y3S_PDG); fitFunc.SetParLimits(11, 10.2, 10.5)
    fitFunc.SetParName(12, "Sigma 3S"); fitFunc.SetParameter(12, 0.06);      fitFunc.SetParLimits(12, 0.01, 0.15)

    fitFunc.SetParName(13, "Bkg Norm");  fitFunc.SetParameter(13, 10)
    fitFunc.SetParName(14, "Bkg Slope"); fitFunc.SetParameter(14, -0.1)

    # 4. Perform Fit 
    print("Fitting with Double-Sided Crystal Ball model...")
    # ---> The "0" flag prevents ROOT from drawing and creating a ghost Canvas! <---
    hInvMass.Fit("fitFunc", "RM0") 

    # 5. Extract Individual Components
    f1S = ROOT.TF1("f1S", DoubleSidedCrystalBall, 8.0, 12.0, 7)
    f1S.SetParameters(fitFunc.GetParameter(0), fitFunc.GetParameter(1), fitFunc.GetParameter(2),
                      fitFunc.GetParameter(3), fitFunc.GetParameter(4), fitFunc.GetParameter(5), fitFunc.GetParameter(6))

    f2S = ROOT.TF1("f2S", DoubleSidedCrystalBall, 8.0, 12.0, 7)
    f2S.SetParameters(fitFunc.GetParameter(7), fitFunc.GetParameter(8), fitFunc.GetParameter(9),
                      fitFunc.GetParameter(3), fitFunc.GetParameter(4), fitFunc.GetParameter(5), fitFunc.GetParameter(6))

    f3S = ROOT.TF1("f3S", DoubleSidedCrystalBall, 8.0, 12.0, 7)
    f3S.SetParameters(fitFunc.GetParameter(10), fitFunc.GetParameter(11), fitFunc.GetParameter(12),
                      fitFunc.GetParameter(3), fitFunc.GetParameter(4), fitFunc.GetParameter(5), fitFunc.GetParameter(6))

    fBkg = ROOT.TF1("fBkg", Background, 8.0, 12.0, 2)
    fBkg.SetParameters(fitFunc.GetParameter(13), fitFunc.GetParameter(14))

    # --- YIELD CALCULATIONS ---
    bin_width = hInvMass.GetBinWidth(1)
    
    yield_1s = f1S.Integral(8.0, 12.0) / bin_width
    yield_2s = f2S.Integral(8.0, 12.0) / bin_width
    yield_3s = f3S.Integral(8.0, 12.0) / bin_width
    
    print("\n--- Fit Yield Results ---")
    print(f"Y(1S) Yield: {yield_1s:.2f}")
    print(f"Y(2S) Yield: {yield_2s:.2f}")
    print(f"Y(3S) Yield: {yield_3s:.2f}")
    print("-------------------------\n")

    # =========================================================================
    # 6. MATPLOTLIB VISUALIZATION
    # =========================================================================
    # Extract ROOT Histogram Data to NumPy
    n_bins = hInvMass.GetNbinsX()
    edges = np.array([hInvMass.GetBinLowEdge(i) for i in range(1, n_bins + 2)])
    counts = np.array([hInvMass.GetBinContent(i) for i in range(1, n_bins + 1)])
    errs = np.array([hInvMass.GetBinError(i) for i in range(1, n_bins + 1)])

    fig, ax = plt.subplots(figsize=(12, 8))
    fig.suptitle(r"$\Upsilon(nS) \rightarrow e^+e^-$ | Invariant Mass Fit", 
                 fontsize=22, fontweight='bold', y=0.96)

    # Plot Histogram Data
    # Plot Histogram Data (Added markersize=4)
    hep.histplot(counts, bins=edges, yerr=errs, histtype='errorbar', 
                 color='black', marker='o', markersize=4, label="Simulation Data", ax=ax, zorder=5)
    
    # Plot Functions
    x_plot = np.linspace(8.0, 12.0, 1000)
    y_tot = np.array([fitFunc.Eval(x) for x in x_plot])
    y_1s  = np.array([f1S.Eval(x) for x in x_plot])
    y_2s  = np.array([f2S.Eval(x) for x in x_plot])
    y_3s  = np.array([f3S.Eval(x) for x in x_plot])
    y_bkg = np.array([fBkg.Eval(x) for x in x_plot])

    # Fills for Signals
    ax.fill_between(x_plot, 0, y_1s, color='dodgerblue', alpha=0.3, label=r"$\Upsilon(1S)$ Signal")
    ax.fill_between(x_plot, 0, y_2s, color='magenta', alpha=0.3, label=r"$\Upsilon(2S)$ Signal")
    ax.fill_between(x_plot, 0, y_3s, color='limegreen', alpha=0.3, label=r"$\Upsilon(3S)$ Signal")

    # Outlines
    ax.plot(x_plot, y_1s, color='dodgerblue', linestyle='--', linewidth=2)
    ax.plot(x_plot, y_2s, color='magenta', linestyle='--', linewidth=2)
    ax.plot(x_plot, y_3s, color='limegreen', linestyle='--', linewidth=2)
    
    ax.plot(x_plot, y_bkg, color='gray', linestyle=':', linewidth=2, label="Background")
    ax.plot(x_plot, y_tot, color='red', linewidth=2, label="Total Fit", zorder=4)

    # Formatting
    ax.set_xlabel(r"Reco $M_{e^+e^-}$ [GeV/$c^2$]", fontsize=16)
    ax.set_ylabel(r"Events / 20 MeV", fontsize=16)
    ax.set_xlim(8.0, 12.0)
    ax.set_ylim(0, peakHeight * 1.4) # Extra headroom for legend
    
    # Formatting
    ax.set_xlabel(r"Reco $M_{e^+e^-}$ [GeV/$c^2$]", fontsize=16)
    ax.set_ylabel(r"Events / 20 MeV", fontsize=16)
    ax.set_xlim(8.0, 12.0)
    
    # --- LOG SCALE UPDATES ---
    ax.set_yscale('log')
    ax.set_ylim(0.5, peakHeight * 50) # Increased top headroom for log scale
    
    draw_epic_labels(ax)
    
    ax.legend(loc="upper right", frameon=False, fontsize=14)
    ax.legend(loc="upper right", bbox_to_anchor=(0.98, 0.98), frameon=False, fontsize=12)
    # Construct the Yields/Fit Text Box
    fit_text = (
        r"$\bf{DSCB\ Fit\ Results}$" + "\n\n"
        r"$\mathbf{\Upsilon(1S)}$" + "\n"
        rf"M = {fitFunc.GetParameter(1):.3f} $\pm$ {fitFunc.GetParError(1):.3f} GeV" + "\n"
        rf"$\sigma$ = {fitFunc.GetParameter(2)*1000.0:.1f} MeV   |   Yield = {int(yield_1s)}" + "\n\n"
        r"$\mathbf{\Upsilon(2S)}$" + "\n"
        rf"M = {fitFunc.GetParameter(8):.3f} $\pm$ {fitFunc.GetParError(8):.3f} GeV" + "\n"
        rf"$\sigma$ = {fitFunc.GetParameter(9)*1000.0:.1f} MeV   |   Yield = {int(yield_2s)}" + "\n\n"
        r"$\mathbf{\Upsilon(3S)}$" + "\n"
        rf"M = {fitFunc.GetParameter(11):.3f} $\pm$ {fitFunc.GetParError(11):.3f} GeV" + "\n"
        rf"$\sigma$ = {fitFunc.GetParameter(12)*1000.0:.1f} MeV   |   Yield = {int(yield_3s)}"
    )

    # Place text box safely on the left (under ePIC labels, where data is flat)
    props = dict(boxstyle='round,pad=0.8', facecolor='white', edgecolor='lightgray', alpha=0.95)
    #ax.text(0.04, 0.65, fit_text, transform=ax.transAxes, fontsize=13, 
    #        verticalalignment='top', bbox=props)
    ax.text(0.74, 0.60, fit_text, transform=ax.transAxes, fontsize=11, 
            verticalalignment='top', bbox=props)

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig("Fit_Inclusive_Upsilon_DSCB_Nice.png", dpi=300)
    print("Saved Fit_Inclusive_Upsilon_DSCB_Nice.png")

if __name__ == "__main__":
    FitUpsilon_Improved()

In [ ]:
import ROOT
import math
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

# Use standard ROOT style for matplotlib
hep.style.use(hep.style.ROOT)

# --- PDG Masses (GeV) for initialization ---
M_Y1S_PDG = 9.46030
M_Y2S_PDG = 10.02326
M_Y3S_PDG = 10.3552

# --- Branching-ratio and state-normalization handling ---
BR_Y1S_EE = 0.0234
BR_Y2S_EE = 0.0191
BR_Y3S_EE = 0.0218

# Visible e+e- eSTARlight cross sections used for the 18x275 samples.
# These are sigma(ep -> e p Upsilon) x BR(Upsilon -> ee), in fb.
TARGET_VISIBLE_XSEC_1S_FB = 327.584
TARGET_VISIBLE_XSEC_2S_FB = 84.780
TARGET_VISIBLE_XSEC_3S_FB = 68.929

EXPECTED_LUMI_INVFB = 10.0
TRUTH_STATE_MASS_WINDOW = 0.20

# Options:
#   "raw" keeps the tuple weights exactly as stored.
#   "target_visible_xsec" rescales each truth state so sum(weight)/L matches the target visible xsec.
STATE_NORM_MODE = "raw"

# Options:
#   "visible_decay" keeps reconstructed e+e- yields.
#   "inclusive_upsilon" additionally multiplies each state by 1 / BR(Upsilon -> ee).
BR_WEIGHT_MODE = "visible_decay"

# --- Double Sided Crystal Ball Function (Crash-Proofed) ---
def DoubleSidedCrystalBall(x, par):
    N = par[0]
    mu = par[1]
    
    # Protect against division by zero
    sigma = abs(par[2])
    if sigma < 1e-6: sigma = 1e-6
        
    alphaL = abs(par[3])
    if alphaL < 1e-6: alphaL = 1e-6
        
    nL = par[4]
    
    alphaH = abs(par[5])
    if alphaH < 1e-6: alphaH = 1e-6
        
    nH = par[6]

    t = (x[0] - mu) / sigma

    try:
        if t < -alphaL:
            A = math.pow(nL / alphaL, nL) * math.exp(-0.5 * alphaL * alphaL)
            B = nL / alphaL - alphaL
            base = B - t
            if base <= 0: return 0.0 # Prevent negative base power errors
            return N * A * math.pow(base, -nL)
            
        elif t > alphaH:
            A = math.pow(nH / alphaH, nH) * math.exp(-0.5 * alphaH * alphaH)
            B = nH / alphaH - alphaH
            base = B + t
            if base <= 0: return 0.0 # Prevent negative base power errors
            return N * A * math.pow(base, -nH)
            
        else:
            return N * math.exp(-0.5 * t * t)
            
    except (ValueError, OverflowError):
        # If the fitter tries extreme values that break Python's math engine, return 0
        return 0.0

# --- Background Function (Crash-Proofed) ---
def Background(x, par):
    try:
        return par[0] * math.exp(par[1] * x[0])
    except OverflowError:
        return 0.0

# --- Total Fit Function ---
def TotalFitFunction(x, par):
    par1S = [par[0], par[1], par[2], par[3], par[4], par[5], par[6]]
    y1s = DoubleSidedCrystalBall(x, par1S)

    par2S = [par[7], par[8], par[9], par[3], par[4], par[5], par[6]]
    y2s = DoubleSidedCrystalBall(x, par2S)

    par3S = [par[10], par[11], par[12], par[3], par[4], par[5], par[6]]
    y3s = DoubleSidedCrystalBall(x, par3S)
    
    bkg = Background(x, [par[13], par[14]])

    return y1s + y2s + y3s + bkg

def draw_epic_labels(ax):
    """Draws the standard ePIC performance block in the top left corner."""
    ax.text(0.04, 0.94, "ePIC Performance", transform=ax.transAxes, 
            fontsize=12, fontweight='bold', ha='left', va='top')
    ax.text(0.04, 0.88, "26.03.1 Campaign", transform=ax.transAxes, 
            fontsize=11, ha='left', va='top')
    ax.text(0.04, 0.82, r"e+p $\Upsilon_{\gamma p}$ 18x275 GeV$^2$", transform=ax.transAxes, 
            fontsize=11, ha='left', va='top')
    ax.text(0.04, 0.76, r"$\mathcal{L}_{proj} = 10$ fb$^{-1}$, $\sqrt{s} = 141$ GeV", transform=ax.transAxes, 
            fontsize=11, ha='left', va='top')


def _truth_state_filter(mass, window=TRUTH_STATE_MASS_WINDOW):
    return f"true_mass >= {mass - window:.12g} && true_mass <= {mass + window:.12g}"

def add_fit_weight_columns(df):
    """Define fit_weight = weight * state_norm * br_corr.

    state_norm optionally rescales each generated Upsilon state to the chosen visible
    eSTARlight cross section. br_corr optionally unfolds the dielectron branching ratio.
    """

    # Default: keep tuple weights.
    scale_1s = 1.0
    scale_2s = 1.0
    scale_3s = 1.0

    if STATE_NORM_MODE == "target_visible_xsec":
        sumw_1s = df.Filter(_truth_state_filter(M_Y1S_PDG)).Sum("weight").GetValue()
        sumw_2s = df.Filter(_truth_state_filter(M_Y2S_PDG)).Sum("weight").GetValue()
        sumw_3s = df.Filter(_truth_state_filter(M_Y3S_PDG)).Sum("weight").GetValue()

        target_1s = TARGET_VISIBLE_XSEC_1S_FB * EXPECTED_LUMI_INVFB
        target_2s = TARGET_VISIBLE_XSEC_2S_FB * EXPECTED_LUMI_INVFB
        target_3s = TARGET_VISIBLE_XSEC_3S_FB * EXPECTED_LUMI_INVFB

        scale_1s = target_1s / sumw_1s if sumw_1s > 0 else 1.0
        scale_2s = target_2s / sumw_2s if sumw_2s > 0 else 1.0
        scale_3s = target_3s / sumw_3s if sumw_3s > 0 else 1.0

        print("\n--- State normalization factors ---")
        print(f"1S: raw xsec = {sumw_1s / EXPECTED_LUMI_INVFB:.3f} fb, target = {TARGET_VISIBLE_XSEC_1S_FB:.3f} fb, scale = {scale_1s:.6g}")
        print(f"2S: raw xsec = {sumw_2s / EXPECTED_LUMI_INVFB:.3f} fb, target = {TARGET_VISIBLE_XSEC_2S_FB:.3f} fb, scale = {scale_2s:.6g}")
        print(f"3S: raw xsec = {sumw_3s / EXPECTED_LUMI_INVFB:.3f} fb, target = {TARGET_VISIBLE_XSEC_3S_FB:.3f} fb, scale = {scale_3s:.6g}")
        print("-----------------------------------\n")
    elif STATE_NORM_MODE != "raw":
        raise ValueError(f"Unknown STATE_NORM_MODE: {STATE_NORM_MODE}")

    state_norm_expr = (
        f"(fabs(true_mass - {M_Y1S_PDG}) < {TRUTH_STATE_MASS_WINDOW}) ? ({scale_1s}) : "
        f"(fabs(true_mass - {M_Y2S_PDG}) < {TRUTH_STATE_MASS_WINDOW}) ? ({scale_2s}) : "
        f"(fabs(true_mass - {M_Y3S_PDG}) < {TRUTH_STATE_MASS_WINDOW}) ? ({scale_3s}) : "
        "1.0"
    )

    if BR_WEIGHT_MODE == "inclusive_upsilon":
        br_corr_expr = (
            f"(fabs(true_mass - {M_Y1S_PDG}) < {TRUTH_STATE_MASS_WINDOW}) ? (1.0/{BR_Y1S_EE}) : "
            f"(fabs(true_mass - {M_Y2S_PDG}) < {TRUTH_STATE_MASS_WINDOW}) ? (1.0/{BR_Y2S_EE}) : "
            f"(fabs(true_mass - {M_Y3S_PDG}) < {TRUTH_STATE_MASS_WINDOW}) ? (1.0/{BR_Y3S_EE}) : "
            "1.0"
        )
    elif BR_WEIGHT_MODE == "visible_decay":
        br_corr_expr = "1.0"
    else:
        raise ValueError(f"Unknown BR_WEIGHT_MODE: {BR_WEIGHT_MODE}")

    print(f"Using STATE_NORM_MODE = {STATE_NORM_MODE}")
    print(f"Using BR_WEIGHT_MODE = {BR_WEIGHT_MODE}")

    return (
        df.Define("state_norm", state_norm_expr)
          .Define("br_corr", br_corr_expr)
          .Define("fit_weight", "weight * state_norm * br_corr")
    )


def FitUpsilon_Improved():
    # 1. Load Data using RDataFrame (Exclusive tuples)
    filename = "~/merged_elecs_test_tuples_fix2.root"
    
    try:
        df = ROOT.RDataFrame("UpsilonTree", filename)
    except Exception as e:
        print(f"Error opening file {filename}: {e}")
        return

    # Define the weight used by the mass-fit histograms.
    # This is the only physics-weight change relative to the original notebook.
    df = add_fit_weight_columns(df)

    # Define Rapidity Bins
    rapidity_bins = [(-1.0, 0.0), (0.0, 1.0)]

    # Setup the 1x2 Matplotlib Figure
    fig, axs = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle(r"$\Upsilon(nS) \rightarrow e^+e^-$ | Invariant Mass Fit by Rapidity", 
                 fontsize=24, fontweight='bold', y=0.98)

    for i, (y_min, y_max) in enumerate(rapidity_bins):
        ax = axs[i]
        
        print(f"\n=======================================================")
        print(f"Processing Rapidity Bin: {y_min} <= y < {y_max}")
        print(f"=======================================================")
        
        # Construct the selection string for this specific bin
        cut_string = (f"reco_mass >= 8.0 && reco_mass <= 12.0 && "
                      f"abs(sigma_epz - 36.0) <= 3.6 && reco_W >= 0.0 && "
                      f"reco_y >= {y_min} && reco_y < {y_max}")
        
        df_bin = df.Filter(cut_string)

        # Generate Histogram dynamically
        hModel = ROOT.RDF.TH1DModel(f"h_invMass_{i}", "Title", 200, 8.0, 12.0)
        hInvMass_ptr = df_bin.Histo1D(hModel, "reco_mass", "fit_weight")
        hInvMass = hInvMass_ptr.GetValue()

        if not hInvMass or hInvMass.GetEntries() == 0:
            print(f"Warning: Histogram is empty for bin {y_min} <= y < {y_max}. Skipping...")
            continue

        # Setup Fit Function
        fitFunc = ROOT.TF1(f"fitFunc_{i}", TotalFitFunction, 8.0, 12.0, 15)
        peakHeight = hInvMass.GetMaximum()
        
        # Initialization
        fitFunc.SetParName(3, "Alpha L"); fitFunc.SetParameter(3, 1.2); fitFunc.SetParLimits(3, 0.1, 5.0)
        fitFunc.SetParName(4, "n L");     fitFunc.SetParameter(4, 2.0); fitFunc.SetParLimits(4, 1.0, 20.0)
        fitFunc.SetParName(5, "Alpha H"); fitFunc.SetParameter(5, 1.5); fitFunc.SetParLimits(5, 0.1, 5.0)
        fitFunc.SetParName(6, "n H");     fitFunc.SetParameter(6, 2.0); fitFunc.SetParLimits(6, 1.0, 20.0)

        fitFunc.SetParName(0, "Norm 1S");  fitFunc.SetParameter(0, peakHeight)
        fitFunc.SetParName(1, "Mean 1S");  fitFunc.SetParameter(1, M_Y1S_PDG); fitFunc.SetParLimits(1, 9.3, 9.6)
        fitFunc.SetParName(2, "Sigma 1S"); fitFunc.SetParameter(2, 0.05);      fitFunc.SetParLimits(2, 0.01, 0.15)

        fitFunc.SetParName(7, "Norm 2S");  fitFunc.SetParameter(7, peakHeight * 0.5) 
        fitFunc.SetParName(8, "Mean 2S");  fitFunc.SetParameter(8, M_Y2S_PDG); fitFunc.SetParLimits(8, 9.9, 10.15)
        fitFunc.SetParName(9, "Sigma 2S"); fitFunc.SetParameter(9, 0.06);      fitFunc.SetParLimits(9, 0.01, 0.15)

        fitFunc.SetParName(10, "Norm 3S");  fitFunc.SetParameter(10, peakHeight * 0.3) 
        fitFunc.SetParName(11, "Mean 3S");  fitFunc.SetParameter(11, M_Y3S_PDG); fitFunc.SetParLimits(11, 10.2, 10.5)
        fitFunc.SetParName(12, "Sigma 3S"); fitFunc.SetParameter(12, 0.06);      fitFunc.SetParLimits(12, 0.01, 0.15)

        fitFunc.SetParName(13, "Bkg Norm");  fitFunc.SetParameter(13, 10)
        # Added a limit to the background slope so it doesn't wander into massive positive exponential blowups
        fitFunc.SetParName(14, "Bkg Slope"); fitFunc.SetParameter(14, -0.1); fitFunc.SetParLimits(14, -10.0, 1.0)

        # Perform Fit (Pass the object `fitFunc` directly, not a string)
        hInvMass.Fit(fitFunc, "RM0") 

        # Extract Individual Components
        f1S = ROOT.TF1(f"f1S_{i}", DoubleSidedCrystalBall, 8.0, 12.0, 7)
        f1S.SetParameters(fitFunc.GetParameter(0), fitFunc.GetParameter(1), fitFunc.GetParameter(2),
                          fitFunc.GetParameter(3), fitFunc.GetParameter(4), fitFunc.GetParameter(5), fitFunc.GetParameter(6))

        f2S = ROOT.TF1(f"f2S_{i}", DoubleSidedCrystalBall, 8.0, 12.0, 7)
        f2S.SetParameters(fitFunc.GetParameter(7), fitFunc.GetParameter(8), fitFunc.GetParameter(9),
                          fitFunc.GetParameter(3), fitFunc.GetParameter(4), fitFunc.GetParameter(5), fitFunc.GetParameter(6))

        f3S = ROOT.TF1(f"f3S_{i}", DoubleSidedCrystalBall, 8.0, 12.0, 7)
        f3S.SetParameters(fitFunc.GetParameter(10), fitFunc.GetParameter(11), fitFunc.GetParameter(12),
                          fitFunc.GetParameter(3), fitFunc.GetParameter(4), fitFunc.GetParameter(5), fitFunc.GetParameter(6))

        fBkg = ROOT.TF1(f"fBkg_{i}", Background, 8.0, 12.0, 2)
        fBkg.SetParameters(fitFunc.GetParameter(13), fitFunc.GetParameter(14))

        # YIELD CALCULATIONS
        bin_width = hInvMass.GetBinWidth(1)
        yield_1s = f1S.Integral(8.0, 12.0) / bin_width
        yield_2s = f2S.Integral(8.0, 12.0) / bin_width
        yield_3s = f3S.Integral(8.0, 12.0) / bin_width
        
        print(f"Y(1S) Yield: {yield_1s:.2f} | Y(2S): {yield_2s:.2f} | Y(3S): {yield_3s:.2f}")

        # MATPLOTLIB VISUALIZATION
        n_bins = hInvMass.GetNbinsX()
        edges = np.array([hInvMass.GetBinLowEdge(j) for j in range(1, n_bins + 2)])
        counts = np.array([hInvMass.GetBinContent(j) for j in range(1, n_bins + 1)])
        errs = np.array([hInvMass.GetBinError(j) for j in range(1, n_bins + 1)])

        hep.histplot(counts, bins=edges, yerr=errs, histtype='errorbar', 
                     color='black', marker='o', markersize=4, label="Simulation Data", ax=ax, zorder=5)
        
        x_plot = np.linspace(8.0, 12.0, 1000)
        y_tot = np.array([fitFunc.Eval(x) for x in x_plot])
        y_1s  = np.array([f1S.Eval(x) for x in x_plot])
        y_2s  = np.array([f2S.Eval(x) for x in x_plot])
        y_3s  = np.array([f3S.Eval(x) for x in x_plot])
        y_bkg = np.array([fBkg.Eval(x) for x in x_plot])

        ax.fill_between(x_plot, 0, y_1s, color='dodgerblue', alpha=0.3, label=r"$\Upsilon(1S)$ Signal")
        ax.fill_between(x_plot, 0, y_2s, color='magenta', alpha=0.3, label=r"$\Upsilon(2S)$ Signal")
        ax.fill_between(x_plot, 0, y_3s, color='limegreen', alpha=0.3, label=r"$\Upsilon(3S)$ Signal")

        ax.plot(x_plot, y_1s, color='dodgerblue', linestyle='--', linewidth=2)
        ax.plot(x_plot, y_2s, color='magenta', linestyle='--', linewidth=2)
        ax.plot(x_plot, y_3s, color='limegreen', linestyle='--', linewidth=2)
        ax.plot(x_plot, y_bkg, color='gray', linestyle=':', linewidth=2, label="Background")
        ax.plot(x_plot, y_tot, color='red', linewidth=2, label="Total Fit", zorder=4)

        # Formatting
        ax.set_title(rf"Rapidity: ${y_min} \leq y < {y_max}$", fontsize=18, pad=10)
        ax.set_xlabel(r"Reco $M_{e^+e^-}$ [GeV/$c^2$]", fontsize=16)
        ax.set_ylabel(r"Events / 20 MeV", fontsize=16)
        ax.set_xlim(8.0, 12.0)
        
        # Log Scale updates
        ax.set_yscale('log')
        ax.set_ylim(0.5, peakHeight * 50) 
        
        draw_epic_labels(ax)
        
        # Main Legend - Positioned Top Right
        ax.legend(loc="upper right", bbox_to_anchor=(0.98, 0.98), frameon=False, fontsize=12)

        # Fit Stats Text Box - Positioned on the Right, Below the Legend
        fit_text = (
            r"$\bf{DSCB\ Fit\ Results}$" + "\n\n"
            r"$\mathbf{\Upsilon(1S)}$" + "\n"
            rf"M = {fitFunc.GetParameter(1):.3f} $\pm$ {fitFunc.GetParError(1):.3f} GeV" + "\n"
            rf"$\sigma$ = {fitFunc.GetParameter(2)*1000.0:.1f} MeV   |   Yield = {int(yield_1s)}" + "\n\n"
            r"$\mathbf{\Upsilon(2S)}$" + "\n"
            rf"M = {fitFunc.GetParameter(8):.3f} $\pm$ {fitFunc.GetParError(8):.3f} GeV" + "\n"
            rf"$\sigma$ = {fitFunc.GetParameter(9)*1000.0:.1f} MeV   |   Yield = {int(yield_2s)}" + "\n\n"
            r"$\mathbf{\Upsilon(3S)}$" + "\n"
            rf"M = {fitFunc.GetParameter(11):.3f} $\pm$ {fitFunc.GetParError(11):.3f} GeV" + "\n"
            rf"$\sigma$ = {fitFunc.GetParameter(12)*1000.0:.1f} MeV   |   Yield = {int(yield_3s)}"
        )

        props = dict(boxstyle='round,pad=0.8', facecolor='white', edgecolor='lightgray', alpha=0.95)
        ax.text(0.68, 0.60, fit_text, transform=ax.transAxes, fontsize=11, 
                verticalalignment='top', bbox=props)

    # Save Output
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    outname = f"Fit_Inclusive_Upsilon_RapidityBins_{STATE_NORM_MODE}_{BR_WEIGHT_MODE}.png"
    fig.savefig(outname, dpi=300)
    print(f"\nSaved {outname}")

if __name__ == "__main__":
    FitUpsilon_Improved()